In [1]:
import os
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from datetime import datetime
import sys
sys.path.append(os.path.abspath("../../"))
from utils.soa_helpers import prepare_global_data, evaluate_model_global, generate_shap_summary

import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv()

ROOT_DIR = os.getenv("ROOT_DIR")

/Users/pasti/e-learning-dropout/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
TRAIN_PATH = os.path.join(ROOT_DIR, ".data/train_test/train_timeseries.csv")
TEST_PATH = os.path.join(ROOT_DIR, ".data/train_test/test_timeseries.csv")
OUTPUT_DIR = os.path.join(ROOT_DIR, "outputs/runs_soa")
MODELS_DIR = os.path.join(OUTPUT_DIR, "models")
PLOTS_DIR = os.path.join(OUTPUT_DIR, "plots")

for path in [OUTPUT_DIR, MODELS_DIR, PLOTS_DIR]:
    os.makedirs(path, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
LAGS = [7, 14, 30]
USE_MACRO = True
TIME_AGG = 'flatten'

CLASSIFIERS = {
    'LogisticRegression': {
        'estimator': LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
        'params': {
            'clf__penalty': ['l2', 'l1'],
            'clf__C': [0.01, 0.1, 1.0, 10.0], 
            'clf__solver': ['liblinear']}
    },
    'RandomForest': {
        'estimator': RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced'),
        'params': {
            'scaler': ['passthrough'],
            'clf__n_estimators': [200, 500], 
            'clf__max_depth': [10, 15, None], 
            'clf__min_samples_leaf': [1, 2],
            'clf__max_features': ['sqrt', 'log2']}
    }
}

In [4]:
results_list = []

for current_lag in LAGS:
    print(f"[INFO] Pipeline for lag {current_lag} days")

    print("[INFO] Processing train set...")
    X_train, y_train, y_strat_train = prepare_global_data(
        TRAIN_PATH, lag=current_lag, use_macro=USE_MACRO, time_aggregation=TIME_AGG
    )
    
    print("[INFO] Processing test set...")
    X_test, y_test, _ = prepare_global_data(
        TEST_PATH, lag=current_lag, use_macro=USE_MACRO, time_aggregation=TIME_AGG
    )

    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

    print(f"[INFO] Final shape: \nTrain {X_train.shape} | Test: {X_test.shape}")

    for algo_name, model_config in CLASSIFIERS.items():
        print(f"[INFO] Training Model: {algo_name}...")
        
        metrics, best_model = evaluate_model_global(
            model_config, X_train, y_train, y_strat_train, X_test, y_test
        )
        
        print(f"[INFO] Results: \nPR-AUC: {metrics['pr_auc']:.4f} | ROC-AUC: {metrics['roc_auc']:.4f}")

        model_path = os.path.join(MODELS_DIR, f"{algo_name}_lag{current_lag}.pkl")
        joblib.dump(best_model, model_path)
        print(f"[INFO] Model saved in {model_path}")
        
        generate_shap_summary(best_model, X_train, X_test, algo_name, current_lag, PLOTS_DIR)

        result_row = {
            'lag': current_lag,
            'algorithm': algo_name,
            'use_macro': USE_MACRO,
            'time_agg': TIME_AGG,
            'accuracy': metrics['accuracy'],
            'precision': metrics['precision'],
            'recall': metrics['recall'],
            'f1': metrics['f1'],
            'roc_auc': metrics['roc_auc'],
            'pr_auc': metrics['pr_auc']
        }
        results_list.append(result_row)

    if results_list:
        df_results = pd.DataFrame(results_list)
        output_csv = os.path.join(OUTPUT_DIR, f"soa_metrics_{TIME_AGG}_{datetime.now()}.csv")
        df_results.to_csv(output_csv, index=False)
        print(f"[INFO] Succeded, saved in: {output_csv}")

[INFO] Pipeline for lag 7 days
[INFO] Processing train set...
[INFO] Processing test set...
[INFO] Final shape: 
Train (4124, 70) | Test: (1032, 70)
[INFO] Training Model: LogisticRegression...
[INFO] Results: 
PR-AUC: 0.8093 | ROC-AUC: 0.5239
[INFO] Model saved in /Users/pasti/e-learning-dropout/outputs/runs_soa/models/LogisticRegression_lag7.pkl


100%|██████████| 200/200 [00:00<00:00, 256.73it/s]


[INFO] Training Model: RandomForest...
[INFO] Results: 
PR-AUC: 0.9095 | ROC-AUC: 0.7253
[INFO] Model saved in /Users/pasti/e-learning-dropout/outputs/runs_soa/models/RandomForest_lag7.pkl


100%|██████████| 200/200 [00:16<00:00, 12.21it/s]


[INFO] Succeded, saved in: /Users/pasti/e-learning-dropout/outputs/runs_soa/soa_7_flatten.csv
[INFO] Pipeline for lag 14 days
[INFO] Processing train set...
[INFO] Processing test set...
[INFO] Final shape: 
Train (4124, 140) | Test: (1032, 140)
[INFO] Training Model: LogisticRegression...
[INFO] Results: 
PR-AUC: 0.8278 | ROC-AUC: 0.5656
[INFO] Model saved in /Users/pasti/e-learning-dropout/outputs/runs_soa/models/LogisticRegression_lag14.pkl


...

[INFO] Succeded, saved in: /Users/pasti/e-learning-dropout/outputs/runs_soa/soa_14_flatten.csv
[INFO] Pipeline for lag 30 days
[INFO] Processing train set...
[INFO] Processing test set...
[INFO] Final shape: 
Train (4124, 300) | Test: (1032, 300)
[INFO] Training Model: LogisticRegression...
[INFO] Results: 
PR-AUC: 0.8314 | ROC-AUC: 0.5814
[INFO] Model saved in /Users/pasti/e-learning-dropout/outputs/runs_soa/models/LogisticRegression_lag30.pkl


100%|██████████| 200/200 [00:20<00:00,  9.53it/s]


[INFO] Training Model: RandomForest...
[INFO] Results: 
PR-AUC: 0.9207 | ROC-AUC: 0.7523
[INFO] Model saved in /Users/pasti/e-learning-dropout/outputs/runs_soa/models/RandomForest_lag30.pkl


100%|██████████| 200/200 [01:22<00:00,  2.44it/s]


[INFO] Succeded, saved in: /Users/pasti/e-learning-dropout/outputs/runs_soa/soa_30_flatten.csv
